# ML S4 · Notebook 03 — The Rest of the Map, Multiclass Mechanics, and Tuning

| | |
|---|---|
| **Session ID** | ML S4 · Notebook 03 of 04 |
| **Course position** | Machine Learning — Session 4 of 10 (Classification &amp; the Model Family Map) |
| **Block types** | ⚙️ **Delivery block** — AI assistance expected throughout. The assessed skill is specifying what you want and verifying you got it. |
| **Prerequisites** | **Notebook 01** (problem shapes) · **Notebook 02** (logistic regression, trees) · ML S3 (Pipeline, StratifiedKFold, cross_val_score) |
| **Connects back to** | **ML S2** — the scaling you did by hand · **ML S3 NB02** — the note that said to use `Pipeline`, not `make_pipeline`, because you would want the step names in S4. This is that notebook. |
| **Connects forward to** | **ML S5** (tuning ensembles) · **ML S6** (tuning under imbalance) · **ML S7** (KNN's distance idea returns for clustering) · **ML S8** (nearest neighbours becomes vector search) · **DL S1** (softmax) |
| **CEP linkage** | Employee Turnover requires a tuned comparison across three model families. The tuning machinery here is what you will use for it. |
| **Run requirements** | `pandas`, `numpy`, `scikit-learn`. Run `ML_S4_00_dataset.ipynb` first. |
| **Checkpoint file** | `subscription_churn.csv` |


## Learning Objectives

By the end of this notebook you will be able to:

1. **State what KNN, Naive Bayes and SVM each assume**, and predict from those assumptions when each will do badly.
2. **Recognise which of them needs feature scaling and why**, without memorising a table.
3. **Choose a sensible shortlist of models for a new problem** in under a minute, and justify the shortlist.
4. **Explain how a two-class algorithm is made to handle three or more classes**, and say what one-vs-rest costs.
5. **Run `GridSearchCV` and `RandomizedSearchCV` over a `Pipeline`**, using the double-underscore syntax to reach nested parameters.
6. **Explain why random search usually beats grid search** on the same budget.
7. **Avoid the tuning mistake that silently inflates every result you report** — and say what the fix costs.

---

## Table of Contents

| § | Section | Type |
|---|---|---|
| 1 | Why bother with a map | Framing |
| 2 | K-nearest neighbours | Core |
| 3 | Naive Bayes | Core |
| 4 | Support vector machines | Core |
| 5 | The map, assembled | Reference |
| 6 | Multiclass mechanics: one-vs-rest and one-vs-one | Core |
| 7 | Hyperparameter tuning: grid, random, and the `__` syntax | ⚙️ Delivery |
| 8 | The mistake that inflates every number you report | ⚙️ Delivery |
| — | Common Pitfalls · FAQ · Conclusion · Transition | Wrap-up |


In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     StratifiedKFold, GridSearchCV,
                                     RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

churn = pd.read_csv('subscription_churn.csv')
X = churn.drop(columns='churned')
y = churn['churned']

cat_cols = ['contract_type', 'region']
num_cols = [c for c in X.columns if c not in cat_cols]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"train {X_tr.shape}   test {X_te.shape}   positive rate {y.mean():.1%}")

train (9600, 12)   test (2400, 12)   positive rate 19.9%


# 1. Why bother with a map

You now know two model families properly. There are dozens more, and a course could spend six hours parading through them.

This notebook spends about half an hour, deliberately, and here is the reasoning.

**In 2026, you will not implement these.** You will call them. The API is four lines and identical across all of them. What you actually need is the ability to answer, in a planning meeting, *"which two or three should we try first, and why?"* — and that requires knowing **what each one assumes**, not how it is derived.

The assumption is the useful part, because the assumption tells you when it will fail. A model whose assumption your data violates will not error. It will fit, report a number, and be quietly wrong.

So for each of the three families below: the intuition, the assumption, the failure mode, and the API. No kernel mathematics, no Bayes derivation.

# 2. K-nearest neighbours

## The intuition

There is no training. KNN memorises the training set and, when asked about a new customer, **finds the k most similar customers it has seen and takes a vote**.

That is the whole algorithm. `k=5` means "look at the five most similar customers; if three of them churned, predict churn with probability 0.6."

## The assumption

**Similar inputs have similar outputs**, where "similar" means *close in distance*. If two customers have nearby feature values, they should behave alike.

That is often reasonable and occasionally disastrous.

## Why scaling is not optional here

Distance is computed across all features at once. If `total_charges` runs to thousands and `satisfaction_score` runs 1–10, then **the distance is essentially the difference in total charges** and satisfaction contributes nothing measurable. The model is not weighing your features; it is being bullied by whichever one has the largest units.

This is the clearest scaling dependency in the whole course, and it is worth measuring rather than accepting on faith.

In [2]:
# KNN with and without scaling. Same k, same data, same split.
knn_scaled = Pipeline([
    ('prep', prep),
    ('model', KNeighborsClassifier(n_neighbors=15)),
])

prep_raw = ColumnTransformer([
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols),
])
knn_raw = Pipeline([
    ('prep', prep_raw),
    ('model', KNeighborsClassifier(n_neighbors=15)),
])

for name, model in [('KNN, features SCALED', knn_scaled),
                    ('KNN, features RAW   ', knn_raw)]:
    model.fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
    acc = model.score(X_te, y_te)
    print(f"{name}:  test accuracy {acc:.4f}   test ROC-AUC {auc:.4f}")

print(f"\nMajority-class floor: {1 - y_te.mean():.4f}")
print("\n--- why: the feature ranges are wildly different ---")
print(X_tr[num_cols].agg(['min', 'max', 'std']).T.to_string(
    float_format=lambda v: f"{v:,.2f}"))

KNN, features SCALED:  test accuracy 0.8433   test ROC-AUC 0.8179
KNN, features RAW   :  test accuracy 0.7992   test ROC-AUC 0.7145

Majority-class floor: 0.8017

--- why: the feature ranges are wildly different ---
                         min      max      std
tenure_months           1.00    72.00    18.27
monthly_charges        19.00   125.00    25.04
total_charges          20.12 9,537.16 1,556.42
num_support_tickets_6m  0.00    11.00     1.37
avg_monthly_usage_gb    5.00   594.20    93.96
late_payments_12m       0.00     6.00     0.88
has_premium_support     0.00     1.00     0.47
num_services            1.00     6.00     1.70
age                    19.00    78.00    17.41
satisfaction_score      1.00    10.00     1.96


Look at the range column. `total_charges` spans thousands; `satisfaction_score` spans nine points. Unscaled, the distance calculation is almost entirely a statement about billing totals — and the accuracy gap between the two rows is the price of that.

## Choosing k

- **Small k** (1, 3) — flexible, follows the training data closely, high variance. `k=1` will classify every training point perfectly and generalise poorly.
- **Large k** (100+) — smooth, stable, high bias. Push it far enough and every prediction converges on the overall base rate.

Same bias–variance trade you met with tree depth, different knob. Cross-validate it.

## The failure mode that matters: dimensionality

KNN degrades badly as the number of features grows. The reason is not obvious and is worth seeing rather than being told: **in high dimensions, everything becomes roughly equidistant from everything else.** The nearest neighbour stops being meaningfully nearer than the average point, and "find the most similar customers" stops meaning anything.

In [3]:
# The curse of dimensionality, measured directly.
rng = np.random.default_rng(3)
print("Random points in a cube. How much closer is the nearest than the farthest?\n")
print(f"{'dimensions':>11} {'nearest':>9} {'farthest':>9} {'ratio':>8}")
for d in [2, 5, 10, 50, 200, 1000]:
    pts = rng.random((500, d))
    query = rng.random(d)
    dists = np.sqrt(((pts - query) ** 2).sum(axis=1))
    print(f"{d:>11} {dists.min():>9.3f} {dists.max():>9.3f} "
          f"{dists.min() / dists.max():>8.3f}")

print("\nAs the ratio approaches 1.0, 'nearest' and 'farthest' stop differing.")
print("KNN's entire premise is that the nearest neighbours are special. In 1000")
print("dimensions they are not, and no amount of tuning k repairs that.")

Random points in a cube. How much closer is the nearest than the farthest?

 dimensions   nearest  farthest    ratio
          2     0.033     1.100    0.030
          5     0.271     1.648    0.164
         10     0.666     1.932    0.344
         50     2.073     3.467    0.598
        200     5.067     6.396    0.792
       1000    12.149    13.414    0.906

As the ratio approaches 1.0, 'nearest' and 'farthest' stop differing.
KNN's entire premise is that the nearest neighbours are special. In 1000
dimensions they are not, and no amount of tuning k repairs that.


> ### 🔎 2026 Reality Check
>
> KNN is rarely the final model on tabular data — it is slow at prediction time, because every query compares against the entire training set, and it degrades with dimensionality as above.
>
> But do not file it away. **The idea behind it is one of the most commercially important in the course.** In Session 8 you will embed text into vectors and retrieve the nearest ones by cosine similarity; that is KNN with a different distance and a clever index. In the GenAI course, that retrieval step is the "R" in RAG, and the entire vector-database industry — FAISS, Chroma, Pinecone — exists to do approximate nearest-neighbour lookup at scale.
>
> So the algorithm is a footnote and the idea is a foundation. Sessions 7 and 8 are where it comes back.


# 3. Naive Bayes

## The intuition

Work out, for each class, how likely the observed features would be if the customer belonged to that class — then pick whichever class makes the observation least surprising.

## The assumption, and why it is called naive

It assumes **every feature is independent of every other, given the class.** That is, once you know a customer churned, knowing their tenure tells you nothing about their total charges.

In our data that is flatly false. `total_charges` is literally computed from `tenure_months` and `monthly_charges`. The assumption is not approximately violated; it is violated by construction.

**The name is an admission, not a description of the method's quality.** It is naive on purpose, because the payoff for the simplification is enormous: instead of learning how features interact, it learns each feature's behaviour separately. That makes it extremely fast and workable on tiny datasets.

## The surprising part

It often classifies well **despite** the assumption being wrong. The reason is worth knowing: to get the *class* right you only need the correct class to score higher than the others. You do not need the probabilities to be accurate.

Which produces the characteristic Naive Bayes signature: **decent rankings, badly calibrated probabilities.** It tends to push its outputs towards 0 and 1, sounding far more certain than it is.

In [4]:
nb_model = Pipeline([('prep', prep), ('model', GaussianNB())]).fit(X_tr, y_tr)
lr_model = Pipeline([('prep', prep),
                     ('model', LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)

p_nb = nb_model.predict_proba(X_te)[:, 1]
p_lr = lr_model.predict_proba(X_te)[:, 1]

print(f"{'model':<22} {'accuracy':>9} {'ROC-AUC':>9}")
print(f"{'Naive Bayes':<22} {accuracy_score(y_te, nb_model.predict(X_te)):>9.4f} "
      f"{roc_auc_score(y_te, p_nb):>9.4f}")
print(f"{'Logistic regression':<22} {accuracy_score(y_te, lr_model.predict(X_te)):>9.4f} "
      f"{roc_auc_score(y_te, p_lr):>9.4f}")

print("\n--- the calibration signature ---")
print(f"{'':<22} {'Naive Bayes':>13} {'Logistic reg':>14}")
print(f"{'predictions < 0.01':<22} {(p_nb < 0.01).mean():>12.1%} {(p_lr < 0.01).mean():>13.1%}")
print(f"{'predictions > 0.99':<22} {(p_nb > 0.99).mean():>12.1%} {(p_lr > 0.99).mean():>13.1%}")
print(f"{'in the middle 0.2-0.8':<22} "
      f"{((p_nb > 0.2) & (p_nb < 0.8)).mean():>12.1%} "
      f"{((p_lr > 0.2) & (p_lr < 0.8)).mean():>13.1%}")

print("\n--- what those confident predictions are actually worth ---")
for name, p in [('Naive Bayes', p_nb), ('Logistic reg', p_lr)]:
    very = p > 0.99
    if very.sum():
        print(f"  {name}: {very.sum()} customers given >99% churn probability; "
              f"{y_te.values[very].mean():.1%} actually churned")
    else:
        print(f"  {name}: no customer given >99% churn probability")

model                   accuracy   ROC-AUC
Naive Bayes               0.7742    0.8256
Logistic regression       0.8538    0.8611

--- the calibration signature ---
                         Naive Bayes   Logistic reg
predictions < 0.01            35.0%         11.8%
predictions > 0.99             0.8%          0.0%
in the middle 0.2-0.8         35.1%         33.5%

--- what those confident predictions are actually worth ---
  Naive Bayes: 18 customers given >99% churn probability; 88.9% actually churned
  Logistic reg: no customer given >99% churn probability


Naive Bayes crowds its predictions into the extremes. Logistic regression spreads them across the range.

**If you only need a ranking, that does not matter.** If a downstream decision multiplies the probability by a dollar figure — an expected-value calculation, which is precisely what Session 6 builds — then it matters a great deal, because the number going into the multiplication is wrong.

## Where Naive Bayes still earns its place

Text classification with word-count features, using `MultinomialNB`. Spam filtering, document categorisation, sentiment as a first pass. Thousands of sparse features, a fast fit, a surprisingly strong baseline — and in that setting the independence assumption ("the word *free* appearing tells you nothing about *offer* appearing") is still wrong but much less damaging.

**As a five-minute baseline it is excellent.** As a final model on dense tabular data with correlated features, it rarely is.

# 4. Support vector machines

## The intuition

Logistic regression draws a boundary between the classes. So does an SVM — but it asks a more specific question: **of all the boundaries that separate these classes, which one leaves the widest gap on either side?**

That gap is the **margin**, and maximising it is the whole idea. The intuition for why it helps: a boundary crammed up against your training points is fragile, and one sitting in the middle of the widest available corridor has room to be slightly wrong about new data.

The points sitting on the edge of that corridor are the **support vectors**. They are the only ones that matter — move any other training point and the boundary does not shift at all.

## The kernel, in one paragraph and no mathematics

Some datasets cannot be separated by a straight line at all. The **kernel trick** lets an SVM behave as though it had projected the data into a much higher-dimensional space where a straight boundary *does* work — without ever actually computing that projection.

The practical consequence is what you need: with `kernel='rbf'`, an SVM can draw curved, complex boundaries. `kernel='linear'` gives you a straight one.

**This is named, not derived.** The mathematics requires machinery this course does not assume, and knowing it would not change which line of code you write.

## The two parameters that matter

- **`C`** — how much to penalise misclassified training points. Low `C`: a wide margin, tolerate errors, simpler boundary. High `C`: fit the training data hard, narrow margin, risk overfitting. It is a regularization knob, and it works the same way as the one from Session 2.
- **`gamma`** (rbf only) — how far a single training point's influence reaches. High `gamma`: very local, wiggly boundaries. Low `gamma`: smooth and broad.

## The reason you will usually not reach for it

Training cost. An SVM with an rbf kernel scales roughly between quadratically and cubically in the number of rows. Fine at 10,000 rows. Painful at 100,000. Impractical at a million.

In [5]:
# What SVM training cost actually looks like as data grows.
print(f"{'rows':>8} {'fit seconds':>13} {'support vectors':>17}")
for n_rows in [500, 1000, 2000, 4000, 8000]:
    Xs = X_tr.iloc[:n_rows]
    ys = y_tr.iloc[:n_rows]
    pipe = Pipeline([('prep', prep), ('model', SVC(kernel='rbf', C=1.0))])
    t0 = time.time()
    pipe.fit(Xs, ys)
    elapsed = time.time() - t0
    n_sv = pipe.named_steps['model'].n_support_.sum()
    print(f"{n_rows:>8} {elapsed:>13.3f} {n_sv:>17}")

print("\nNote how the time grows faster than the row count.")
print("Doubling the data more than doubles the fit time -- that is the scaling problem.")

    rows   fit seconds   support vectors
     500         0.133               232
    1000         0.106               401
    2000         0.403               762
    4000         1.546              1509
    8000         7.464              2873

Note how the time grows faster than the row count.
Doubling the data more than doubles the fit time -- that is the scaling problem.


Extrapolate that curve to a few hundred thousand rows and the problem is obvious. `LinearSVC` scales far better and is a reasonable choice when you want the margin idea without the kernel cost — but at that point you are close to logistic regression, which also gives you probabilities.

**One more practical drawback:** an SVM does not naturally produce probabilities. `SVC(probability=True)` will give you some, but it obtains them by fitting a second model on top via cross-validation, which multiplies the training cost and is often poorly calibrated anyway. Given how much of this course rests on probabilities, that is a real cost.

**Scaling is mandatory.** Like KNN, an SVM works in distances. Unscaled features produce a meaningless margin.


### 🖼️ Image Slot — How five model families carve up the same two-dimensional data

**What to show:** A single row of five small square panels, each showing the identical two-class scatter (two interleaving crescent shapes, the classic 'moons' pattern), with the model's decision regions shaded behind the points. Panels in order: Logistic Regression (one straight boundary, visibly failing to separate the crescents), Decision Tree (blocky axis-aligned rectangles), KNN k=15 (smooth curved boundary hugging the data), Naive Bayes (smooth quadratic-looking boundary), SVM rbf (smooth curved boundary with the margin corridor faintly shaded and support vectors circled). Title each panel with the model name and its test accuracy on this toy data.

**Why here:** This is the payoff figure for the whole notebook. Five paragraphs of assumptions collapse into one glance: which families draw straight lines, which draw boxes, which draw smooth curves. Learners retain the picture far longer than the prose, and it makes the comparison table in Section 5 feel like a summary rather than a list to memorise.

**Placement:** Opening of Section 5, before the summary table, as the visual the table then annotates.

**Alt text:** Five panels showing the decision boundaries drawn by logistic regression, a decision tree, KNN, Naive Bayes and an SVM on the same two-crescent dataset.

<!-- INSERT IMAGE: s4_03_five_decision_boundaries.png -->


# 5. The map, assembled

| Model | Core assumption | Needs scaling? | Gives probabilities? | Handles interactions? | Reach for it when |
|---|---|---|---|---|---|
| **Logistic regression** | effects are linear in log-odds, and additive | yes (for stable coefficients and convergence) | yes, well calibrated | no — you must build them | always, as the interpretable baseline |
| **Decision tree** | the target is a series of axis-aligned splits | **no** | yes, but coarse | **yes**, natively | you need a rule a human can read |
| **KNN** | similar inputs → similar outputs | **yes, critically** | yes, from neighbour votes | yes, implicitly | few features, plenty of rows, boundary is irregular |
| **Naive Bayes** | features independent given the class | mild | yes, but **badly calibrated** | no | text, sparse counts, or a five-minute baseline |
| **SVM** | classes separable with the widest possible margin | **yes, critically** | not natively | yes, with a kernel | small-to-medium data, high dimensions, accuracy over interpretability |

## How to actually use this

You do not run all five. A workable default for a new tabular classification problem, and roughly what an experienced practitioner would do:

1. **Majority-class baseline.** Not a model. The number everything else must beat.
2. **Logistic regression.** Fast, interpretable, and if it does nearly as well as anything else, you are probably finished.
3. **A gradient-boosted tree** — Session 5. On tabular data this is usually the strongest thing you will fit.
4. **Anything else only if you have a reason.** A specific assumption that fits, an interpretability constraint, a latency budget.

Notice what that list does *not* include. KNN, Naive Bayes and SVM are on the map so you recognise them, can discuss them, and know when their assumptions happen to match your problem. They are not on the default path for tabular work in 2026.

**Being able to say why you did *not* use something is as much a signal of competence as using it well.**

In [6]:
# All five, same pipeline, same CV, same data. About a minute.
models = {
    'Logistic regression': LogisticRegression(max_iter=2000),
    'Decision tree (d=6)': DecisionTreeClassifier(max_depth=6, min_samples_leaf=40,
                                                  random_state=0),
    'KNN (k=25)':          KNeighborsClassifier(n_neighbors=25),
    'Naive Bayes':         GaussianNB(),
    'SVM (rbf, subset)':   SVC(kernel='rbf', C=1.0, probability=False),
}

print(f"{'model':<24} {'CV accuracy':>12} {'test acc':>10} {'fit sec':>9}")
print(f"{'majority-class floor':<24} {'-':>12} {1 - y_te.mean():>10.4f} {'-':>9}")

for name, est in models.items():
    # The SVM gets a subset; on the full training set it is slow enough to
    # disrupt the session, which is itself the point being made.
    if 'SVM' in name:
        Xf, yf = X_tr.iloc[:4000], y_tr.iloc[:4000]
    else:
        Xf, yf = X_tr, y_tr
    pipe = Pipeline([('prep', prep), ('model', est)])
    t0 = time.time()
    cv_acc = cross_val_score(pipe, Xf, yf, cv=cv, scoring='accuracy').mean()
    pipe.fit(Xf, yf)
    elapsed = time.time() - t0
    print(f"{name:<24} {cv_acc:>12.4f} {pipe.score(X_te, y_te):>10.4f} "
          f"{elapsed:>9.2f}")

model                     CV accuracy   test acc   fit sec
majority-class floor                -     0.8017         -
Logistic regression            0.8517     0.8538      0.95
Decision tree (d=6)            0.8505     0.8512      1.25
KNN (k=25)                     0.8445     0.8454      4.75
Naive Bayes                    0.7697     0.7742      0.45
SVM (rbf, subset)              0.8573     0.8592      5.36


### Read the spread, and then read the cost column

Two things worth noting, and the second is easy to miss.

**The models cluster.** Most of them land within a couple of points of each other, and all of them beat the majority-class floor. That is the normal situation on a reasonable tabular dataset, and it is why Session 3 spent a whole notebook on deciding whether a gap is real. A difference of one point between two of these rows is not a finding.

**The cost column varies far more than the accuracy column.** That is the practically important observation. When two models perform equivalently, the tiebreakers are training time, prediction latency, interpretability and operational simplicity — not the fourth decimal place of an accuracy score.

> ### 🟢 Try It Yourself
>
> Add `RandomForestClassifier(n_estimators=200, random_state=0)` to the dictionary above and re-run it. Before you do, predict where it will land relative to the decision tree, and why.
>
> Then answer: given the results, which model would you actually deploy for a retention team, and what would you say to justify it?

<details>
<summary>Solution</summary>

```python
from sklearn.ensemble import RandomForestClassifier
models['Random forest (200)'] = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=4, random_state=0, n_jobs=-1)
```

The forest should beat the single tree, for the reason Notebook 02 §9 ended on: averaging many trees keeps the interaction-finding while smoothing out the staircase approximation of linear effects. That is Session 5's entire argument, and this is a preview of it rather than a substitute.

On the deployment question there is no single right answer, and that is the point of asking. A defensible one: *"Logistic regression, because it is within a point of everything else, the retention team has to justify each call, and a coefficient table is auditable. If we later find the gap to a boosted model is worth more than the explainability, we revisit."* An equally defensible one argues the opposite — as long as it names the trade rather than just the score.

</details>


# 6. Multiclass mechanics: one-vs-rest and one-vs-one

Notebook 01 told you multiclass exists and what it does to your metrics. This section is the mechanism: **how an algorithm that only knows how to draw one boundary handles three classes.**

## The problem

Logistic regression produces *one* probability. A sigmoid outputs a single number between 0 and 1. That is a two-class device by construction — there is no obvious place to put a third answer.

Support vector machines have the same limitation, and for the same reason: one boundary separates two things.

So there are two ways out, and sklearn picks one for you silently. Knowing which one matters when you read `coef_`, when you count how long training takes, and when you interpret probabilities.

## Strategy 1 — one-vs-rest (OvR)

Train **one binary classifier per class**, each asking a deliberately lopsided question:

- Classifier A: *Month-to-month, or not?*
- Classifier B: *One year, or not?*
- Classifier C: *Two year, or not?*

To predict, run all three and **take the class whose classifier is most confident**.

Three classes means three models. Ten classes means ten. Cost grows linearly.

## Strategy 2 — one-vs-one (OvO)

Train a classifier for **every pair** of classes:

- A vs B, A vs C, B vs C

Each model sees only rows from its two classes. To predict, run all of them and **hold a vote**.

Three classes means 3 models — the same. But ten classes means 45, because the count is $K(K-1)/2$. Cost grows quadratically.

## Which one you get

| Estimator | Default strategy |
|---|---|
| `LogisticRegression` | **Multinomial** — a genuine multiclass fit, not a wrapper |
| `SVC` | One-vs-one |
| `LinearSVC` | One-vs-rest |
| `DecisionTreeClassifier`, `RandomForestClassifier`, `KNeighborsClassifier`, `GaussianNB` | **Natively multiclass** — no wrapper needed at all |

That last row is worth pausing on. Trees, forests, KNN and Naive Bayes never had the two-class limitation. A tree leaf can hold any number of classes; a KNN vote can be three-way. **Only the boundary-drawing models needed a workaround.**

Modern `LogisticRegression` skips the wrapper too, fitting all classes jointly with a **softmax** — the direct generalisation of the sigmoid. You will meet softmax again as the standard output layer of a classification network in **DL S1**. It is the same idea arriving under a different name.

## 6.1 Worked example — toy: resolving three classifiers by hand

Three one-vs-rest classifiers have each scored the same customer. The scores are decision-function values — higher means *more confident this is my class*.

| Classifier | Question | Score |
|---|---|---|
| A | Month-to-month vs rest | **+0.8** |
| B | One year vs rest | −0.3 |
| C | Two year vs rest | −1.2 |

OvR takes the **argmax**: A wins, so the prediction is *Month-to-month*.

Two things to notice, because both cause confusion later:

**The scores do not sum to anything meaningful.** 0.8 − 0.3 − 1.2 = −0.7. That is not a probability distribution, and it was never supposed to be. Each classifier was trained on its own separate question and has no idea the others exist. Getting probabilities out of OvR requires an extra normalisation step, and those probabilities are approximations.

**All three can be negative.** If the scores were −0.2, −0.9, −1.5, OvR still predicts A — the least-rejected class. Every classifier said "not mine" and the algorithm picks the one that said it least emphatically. That is the honest behaviour of the method, not a bug, but it means low confidence can hide behind a confident-looking label.


In [7]:
# Verify the toy by hand, then with sklearn's own machinery.
import numpy as np

scores = {'Month-to-month': 0.8, 'One year': -0.3, 'Two year': -1.2}
print("OvR by hand:", max(scores, key=scores.get))

# The all-negative case
scores2 = {'Month-to-month': -0.2, 'One year': -0.9, 'Two year': -1.5}
print("all-negative case:", max(scores2, key=scores2.get),
      "  <- every classifier said 'not mine'")

OvR by hand: Month-to-month
all-negative case: Month-to-month   <- every classifier said 'not mine'


## 6.2 Worked example — intermediate: where the averaging choice bites

Notebook 01 introduced macro / micro / weighted averaging. Here is the case that shows why it is not a formatting preference.

Build a deliberately skewed three-class problem: 90% class 0, 7% class 1, 3% class 2. Then use a classifier that ignores the two rare classes entirely.


In [8]:
from sklearn.metrics import f1_score, precision_score, recall_score

# 1000 rows: 900 / 70 / 30
y_true = np.array([0]*900 + [1]*70 + [2]*30)

# A model that has quietly given up and always says class 0
y_pred = np.zeros(1000, dtype=int)

for avg in ['micro', 'macro', 'weighted']:
    print(f"{avg:>9} F1 = {f1_score(y_true, y_pred, average=avg, zero_division=0):.3f}")

print()
print("per-class recall:", recall_score(y_true, y_pred, average=None, zero_division=0).round(3))

    micro F1 = 0.900
    macro F1 = 0.316
 weighted F1 = 0.853

per-class recall: [1. 0. 0.]


Read those three numbers carefully, because they describe the same useless model.

**Micro F1 is 0.900.** Micro-averaging pools every prediction into one big count before computing the metric, so the 900 easy rows drown out everything else. On a single-label multiclass problem, micro F1 is mathematically identical to accuracy — which is why it inherits accuracy's blindness to rare classes.

**Weighted F1 is around 0.853.** Weighting by class frequency is a milder version of the same failure. Class 0 carries 90% of the weight, so the model's total failure on classes 1 and 2 barely registers.

**Macro F1 is 0.316.** Macro-averaging computes F1 separately per class and then takes a plain unweighted mean. Class 0 scores about 0.947; classes 1 and 2 both score 0. The average of those three is what you see, and it is the only one of the three numbers that tells you something is badly wrong.

The rule that follows: **if the rare classes are the ones you care about, report macro.** If you report micro on an imbalanced multiclass problem you have essentially reported accuracy while appearing to have done something more careful.

This is the multiclass restatement of the same lesson S3 taught for the binary case, and it is why Notebook 01 put "which average?" in the metrics column of the problem-shape table.


## 6.3 Worked example — realistic: predicting contract type

Our dataset has a genuine three-class column sitting in it: `contract_type`. Predicting it is a real business question — *which customers look like they belong on a longer contract?* — and it lets us watch the mechanics on data you already know.

Note what changes and what does not. The split is still stratified. The pipeline is unchanged. `cross_val_score` is unchanged. **Only the metric choice needs thought.**


In [9]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import pandas as pd

churn_mc = pd.read_csv('subscription_churn.csv')

y_mc = churn_mc['contract_type']              # 3 classes, now the target
X_mc = churn_mc.drop(columns=['contract_type'])

print("class balance:")
print(y_mc.value_counts(normalize=True).round(3).to_string())

cat_mc = ['region']
num_mc = [c for c in X_mc.columns if c not in cat_mc]

Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    X_mc, y_mc, test_size=0.2, random_state=42, stratify=y_mc)

prep_mc = ColumnTransformer([
    ('num', StandardScaler(), num_mc),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_mc),
])

mc_pipe = Pipeline([
    ('prep', prep_mc),
    ('model', LogisticRegression(max_iter=2000)),
])
mc_pipe.fit(Xm_tr, ym_tr)
pred_mc = mc_pipe.predict(Xm_te)

print()
print(classification_report(ym_te, pred_mc, digits=3))

class balance:
contract_type
Month-to-month    0.548
One year          0.265
Two year          0.187

                precision    recall  f1-score   support

Month-to-month      0.737     0.907     0.813      1314
      One year      0.486     0.594     0.535       636
      Two year      0.250     0.002     0.004       450

      accuracy                          0.655      2400
     macro avg      0.491     0.501     0.451      2400
  weighted avg      0.579     0.655     0.588      2400



In [10]:
# What shape did the coefficients come out as?
clf = mc_pipe.named_steps['model']
print("classes_ :", clf.classes_)
print("coef_ shape:", clf.coef_.shape, " <- one ROW PER CLASS, not one row total")
print()
print("Binary logistic regression in Notebook 02 gave coef_ shape (1, n_features).")
print("Three classes gives (3, n_features): each row is that class's own set of weights.")

classes_ : ['Month-to-month' 'One year' 'Two year']
coef_ shape: (3, 14)  <- one ROW PER CLASS, not one row total

Binary logistic regression in Notebook 02 gave coef_ shape (1, n_features).
Three classes gives (3, n_features): each row is that class's own set of weights.


**Reading the report — and there is something alarming in it.**

Look at the **Two year** row. Recall is approximately **0.002**: out of 450 two-year customers in the test set, the model correctly identifies roughly *one*. Its F1 is effectively zero. The model has, for all practical purposes, decided that the Two year class does not exist.

Now look at the accuracy line: **0.655**. Against a most-common-class floor of 0.548, that looks like a model that has learned something. And it has — it discriminates Month-to-month from One year tolerably well. But an accuracy of 0.655 is quietly reporting a model that is **completely blind to one of its three classes**.

This is §6.2's lesson arriving on real data rather than a constructed example. Compare the two averages in the output: weighted F1 is around 0.588, macro F1 around 0.451. The gap between them *is* the failure on Two year. Had you reported only accuracy, or only the weighted average, nothing in your report would have revealed that a third of the business question is going unanswered.

**Why it happens here** is worth naming, because it is not a bug. Two year is the smallest class (19%), and its members look similar to One year customers on most features. Faced with an ambiguous row, the model maximises expected correctness by guessing the more common of the two. That is rational behaviour given what it was asked to optimise — and it is exactly why **the metric you choose changes what the model becomes**, which is the argument Session 6 develops in full.

**Also note:** the classes are unequal (roughly 55 / 27 / 19), so accuracy is carried disproportionately by Month-to-month.

**Reading `coef_`.** In Notebook 02 the binary model had a single row of coefficients, and a positive value meant *pushes towards churn*. With three classes you get one row per class, and a positive value in row *k* means *pushes towards class k specifically*. There is no longer a single "the coefficient for tenure" — there are three, one per class, and they must be interpreted relative to each other.

This is the concrete cost of multiclass that Notebook 01 flagged in the abstract: **interpretation gets harder faster than the code does.** The code barely changed. The story you tell about the coefficients changed a great deal.


### ✅ Quick Check

1. You have 12 classes and choose `SVC`. How many underlying binary models get trained?
2. Your multiclass model reports micro F1 of 0.91 and macro F1 of 0.44. What has almost certainly happened?
3. Which needs no multiclass wrapper at all: a decision tree, or a linear SVM?

<details>
<summary>Answers</summary>

1. **66.** `SVC` defaults to one-vs-one, so $K(K-1)/2 = 12 \times 11 / 2 = 66$. This is exactly why `SVC` becomes painful as classes multiply, on top of its row-count problem from §4.
2. **The model is doing well on common classes and failing on rare ones.** Micro is dominated by the frequent classes; macro gives every class equal say. A gap that large means at least one class is being predicted rarely or never.
3. **The decision tree.** A leaf can hold any number of classes, so nothing needs wrapping. A linear SVM draws one boundary between two groups and needs OvR to handle more.

</details>


# 7. Hyperparameter tuning: grid, random, and the `__` syntax

Every model in this notebook has knobs you have so far left at their defaults: `k` for KNN, `C` for logistic regression and SVM, `max_depth` for trees. Those defaults are reasonable starting points chosen by the library authors. They are not tuned for your problem, and nobody claimed they were.

**Tuning is the search for a better setting, done honestly.** The honesty is the hard part, and §8 is about the specific way it usually goes wrong.

## Parameter vs hyperparameter, stated once

A **parameter** is learned from data during `fit` — the coefficients in logistic regression, the split points in a tree. You never set these.

A **hyperparameter** is set *before* fitting and controls how fitting happens — `C`, `k`, `max_depth`. The model cannot learn these from the training data, because a model asked to choose its own `max_depth` to minimise training error would answer "infinite" every time.

That is the whole reason tuning needs **validation** data rather than training data. It is the S3 lesson arriving in a new costume.

## Reaching into a Pipeline: the payoff of a promise

Session 3 Notebook 02 ended with a note that said: *use `Pipeline` rather than `make_pipeline`, because you will want the named steps in ML S4.* This is that moment.

When your estimator is wrapped in a `Pipeline` and a `ColumnTransformer`, the thing you want to tune is buried several layers down. sklearn addresses nested components with **double underscores**:

```
model__C          -> the step named 'model', its parameter C
prep__num__with_mean  -> step 'prep', its sub-transformer 'num', that object's with_mean
```

Read `__` as *"reach inside"*. Had you used `make_pipeline`, the step would be auto-named `logisticregression` and your grid key would be `logisticregression__C` — which works, but breaks the moment you swap the model, because the name changes with it. Naming your steps `model` and `prep` means **the grid keys stay stable when the estimator changes.** That is why the note was written.

## 7.1 Worked example — toy: a grid with four combinations, inspected

Start deliberately small enough to check by hand: **four combinations**, each evaluated with 5-fold CV, so 20 fits total. The point is not the result — it is seeing exactly what the object does.

The two axes are chosen to demonstrate the `__` syntax at both depths:

- `model__C` — one level down: the step named `model`, its parameter `C`.
- `prep__num__with_mean` — **two levels down**: the step named `prep`, its sub-transformer named `num`, that `StandardScaler`'s `with_mean` parameter.

That second one is the payoff of naming your steps. You are reaching through a `Pipeline`, into a `ColumnTransformer`, into one specific transformer inside it, and setting a parameter — with a string.


In [11]:
from sklearn.model_selection import GridSearchCV

pipe_lr = Pipeline([
    ('prep', prep),
    ('model', LogisticRegression(max_iter=2000)),
])

grid_small = {
    'model__C': [0.01, 1.0],              # reach into the model
    'prep__num__with_mean': [True, False],  # reach into prep -> num -> the scaler
}

gs_small = GridSearchCV(
    pipe_lr, grid_small, cv=cv, scoring='roc_auc', n_jobs=-1, return_train_score=False)
gs_small.fit(X_tr, y_tr)

res = pd.DataFrame(gs_small.cv_results_)[
    ['param_model__C', 'param_prep__num__with_mean', 'mean_test_score', 'std_test_score', 'rank_test_score']
].sort_values('rank_test_score')
print(res.to_string(index=False))
print()
print("best params :", gs_small.best_params_)
print(f"best CV AUC : {gs_small.best_score_:.4f}")
print(f"total fits  : 4 combos x 5 folds = 20")

 param_model__C  param_prep__num__with_mean  mean_test_score  std_test_score  rank_test_score
           1.00                       False         0.852820        0.005936                1
           1.00                        True         0.852788        0.005935                2
           0.01                        True         0.846004        0.005848                3
           0.01                       False         0.845999        0.005843                4

best params : {'model__C': 1.0, 'prep__num__with_mean': False}
best CV AUC : 0.8528
total fits  : 4 combos x 5 folds = 20


Three things in that output deserve attention.

**`mean_test_score` is a cross-validation score, not a test-set score.** The naming is genuinely unfortunate. `GridSearchCV` calls each held-out fold a "test" fold internally. Your actual test set has not been touched and must not be until the very end.

**`std_test_score` is the number most people skip.** If the best configuration scores 0.001 above the second-best while the standard deviation across folds is 0.01, you have not found a better setting — you have found fold-to-fold noise. This is precisely the S3 discipline: a difference is not a difference until you have looked at its spread.

**`best_estimator_` is already refitted.** By default `GridSearchCV` refits the winning configuration on the entire training set once the search finishes, so `gs_small.predict(...)` works immediately.

**And now look at the pattern in the four rows**, because it is about to become the argument of the next section. Sort them mentally by `C`: the two rows with `C=1.0` score noticeably above the two with `C=0.01`. Now sort by `with_mean`: the difference between `True` and `False` is in the fifth decimal place — far smaller than `std_test_score`.

**One of these two hyperparameters matters and the other does not.** The grid spent half its budget discovering that centring the features changes essentially nothing. Hold that thought.


## 7.2 Worked example — intermediate: grid versus random on the same budget

Here is the claim to be tested: **on an equal budget, random search usually beats grid search.** That sounds like it should be false. Grid search is systematic; random search is, apparently, guessing.

The reasoning is about how budget gets *spent*, not about luck.

Suppose you have two hyperparameters, and — as is typical — **one of them matters a great deal and the other barely matters at all.** You do not know in advance which is which.

- A **grid** of 4 × 4 = 16 points tests only **4 distinct values** of the important parameter. The other 12 fits re-test those same 4 values against variations of a parameter that does not matter.
- **Random search with 16 draws** tests **16 distinct values** of the important parameter, because every draw is a fresh value on both axes.

Same compute. Four times the resolution where resolution matters. Below, both searches get 16 fits' worth of budget on the same data.


In [12]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform
import time

pipe_t = Pipeline([('prep', prep), ('model', DecisionTreeClassifier(random_state=0))])

# GRID: 4 x 4 = 16 combinations
grid_t = {
    'model__max_depth':        [3, 5, 8, 12],
    'model__min_samples_leaf':  [1, 5, 20, 50],
}
t0 = time.time()
gs_t = GridSearchCV(pipe_t, grid_t, cv=cv, scoring='roc_auc', n_jobs=-1).fit(X_tr, y_tr)
t_grid = time.time() - t0

# RANDOM: 16 draws from continuous / wide ranges
rand_t = {
    'model__max_depth':        np.arange(2, 21),
    'model__min_samples_leaf':  np.arange(1, 101),
}
t0 = time.time()
rs_t = RandomizedSearchCV(pipe_t, rand_t, n_iter=16, cv=cv, scoring='roc_auc',
                          random_state=0, n_jobs=-1).fit(X_tr, y_tr)
t_rand = time.time() - t0

print(f"GRID    best AUC {gs_t.best_score_:.4f}  params {gs_t.best_params_}   ({t_grid:.1f}s)")
print(f"RANDOM  best AUC {rs_t.best_score_:.4f}  params {rs_t.best_params_}   ({t_rand:.1f}s)")
print()
print(f"distinct max_depth values tried by grid   : {len(set(grid_t['model__max_depth']))}")
print(f"distinct max_depth values tried by random : "
      f"{len(set(d['model__max_depth'] for d in rs_t.cv_results_['params']))}")

GRID    best AUC 0.8465  params {'model__max_depth': 8, 'model__min_samples_leaf': 50}   (4.7s)
RANDOM  best AUC 0.8470  params {'model__min_samples_leaf': np.int64(64), 'model__max_depth': np.int64(9)}   (4.0s)

distinct max_depth values tried by grid   : 4
distinct max_depth values tried by random : 12


Whichever search wins on this particular run, **the resolution line at the bottom is the durable result.** Random search explored substantially more distinct values of `max_depth` for exactly the same number of fits.

An honest caveat, since this course does not manufacture drama: with only two hyperparameters and a well-behaved dataset, the two methods often land in much the same place, and grid search may occasionally win. The advantage of random search **grows with the number of hyperparameters**, because the waste in a grid compounds with every axis you add. At five hyperparameters the grid is testing an enormous number of near-identical configurations, and random search is not.

The practical rule most teams settle on:

- **Few parameters, small discrete sets, and you want reproducible exhaustive coverage** → grid.
- **More than two or three parameters, or continuous ranges, or a fixed compute budget** → random.
- **Serious tuning budget** → Bayesian optimisation (`optuna`), which uses earlier results to decide where to look next. Out of scope here; worth the name.


### 🖼️ Image Slot — Grid search vs random search coverage

**What to show:** Two square panels side by side, each showing the same 2D hyperparameter space with 9 evaluation points. Above each square, a curve showing 'this parameter matters'; to the left, a nearly flat curve showing 'this one does not'. Left panel (grid): points in a 3x3 lattice, with dashed lines dropping to the top curve showing only 3 distinct values sampled. Right panel (random): 9 scattered points, dashed lines showing 9 distinct values sampled on the important axis.

**Why here:** This is the single clearest argument in the section and it is almost impossible to convey in prose. The moment learners see three dropped lines versus nine, the 'same budget, more resolution' claim becomes obvious rather than asserted.

**Placement:** Immediately after the grid-vs-random code output, before the practical rule list.

**Alt text:** Grid search samples few distinct values of the important hyperparameter; random search samples many, for the same number of evaluations.

<!-- INSERT IMAGE: s4_03_grid_vs_random.png -->


## 7.3 Worked example — realistic: tuning across two families, fairly

Now the version you would actually run. Two model families, each with its own grid, both searched with the same CV object and the same scoring metric, then compared.

The fairness conditions matter and are easy to violate by accident:

- **The same `cv` object** — so both are scored on identical folds.
- **The same `scoring`** — AUC here, because the classes are imbalanced and S3 established that accuracy is the wrong lens.
- **The test set untouched by both** — it is not passed to either search.


In [13]:
searches = {}

searches['logistic'] = RandomizedSearchCV(
    Pipeline([('prep', prep), ('model', LogisticRegression(max_iter=3000))]),
    {'model__C': loguniform(1e-3, 1e2)},
    n_iter=20, cv=cv, scoring='roc_auc', random_state=0, n_jobs=-1)

searches['tree'] = RandomizedSearchCV(
    Pipeline([('prep', prep), ('model', DecisionTreeClassifier(random_state=0))]),
    {'model__max_depth': np.arange(2, 21),
     'model__min_samples_leaf': np.arange(1, 101),
     'model__criterion': ['gini', 'entropy']},
    n_iter=20, cv=cv, scoring='roc_auc', random_state=0, n_jobs=-1)

rows = []
for name, s in searches.items():
    s.fit(X_tr, y_tr)
    i = s.best_index_
    rows.append({
        'model': name,
        'CV AUC': round(s.best_score_, 4),
        'fold std': round(s.cv_results_['std_test_score'][i], 4),
        'best params': s.best_params_,
    })

summary = pd.DataFrame(rows).sort_values('CV AUC', ascending=False)
print(summary.to_string(index=False))

   model  CV AUC  fold std                                                                            best params
logistic  0.8529    0.0060                                                       {'model__C': 42.460313017682125}
    tree  0.8508    0.0077 {'model__min_samples_leaf': 65, 'model__max_depth': 15, 'model__criterion': 'entropy'}


### 🔎 2026 Reality Check

Tuning is where a great deal of junior effort goes and where a surprisingly small amount of value is found.

The realistic ranking of what moves a model, in descending order of impact:

1. **Better features** — usually worth more than everything below it combined.
2. **More or cleaner data.**
3. **A better model family** — the logistic-to-boosted-trees jump, which is Session 5.
4. **Hyperparameter tuning** — typically fractions of a point once the above are settled.

None of that makes tuning pointless. It makes tuning **the last thing you do, not the first**. An engineer who spends a day tuning a model built on unexamined features has optimised the wrong layer, and it is a common enough mistake that noticing it is a genuine interview signal.

The counter-case, stated fairly: some hyperparameters are not fine-tuning at all. A tree at `max_depth=None` versus `max_depth=6` is not a marginal adjustment — it is the difference between a memoriser and a model, as Notebook 02 §8 showed. **Regularisation-shaped hyperparameters matter more than the rest**, because they decide overfitting rather than trimming it.


# 8. The mistake that inflates every number you report

Here is a workflow that looks careful and is not:

1. Split into train and test.
2. Run a search on the training set. Get `best_score_` — the CV score of the winning configuration.
3. **Report `best_score_` as the model's expected performance.**

Step 3 is the error, and it is subtle enough that it appears in submitted work regularly.

## Why it inflates

`best_score_` is the **maximum** over everything you tried. Every configuration's CV score contains some noise. Taking the maximum over 20 noisy numbers systematically favours configurations that got lucky on those particular folds — not only configurations that are genuinely good.

The more configurations you search, the more optimistic that maximum becomes. Searching harder makes the number look better while the model does not improve. This is the same mechanism that makes a p-value meaningless after you have tested twenty hypotheses and reported the best one — **you have already met this idea in your statistics background under multiple comparisons.**

The validation folds were **spent** on selection. They cannot also serve as an unbiased estimate of performance.

Three examples below, on a ladder. The first isolates the effect. The second looks for it on our full dataset and — instructively — fails to find it. The third shows the regime where it becomes serious.

## 8.1 Worked example — toy: the selection effect, isolated

Forget the test set for a moment. Look *only* inside the search, at the 20 configurations it evaluated.

If `best_score_` were an honest estimate of "how good is a well-configured tree," it should sit near the middle of a cluster of good configurations. Instead it sits at the top by construction, because that is what a maximum is.


In [14]:
search = searches['tree']
scores = search.cv_results_['mean_test_score']

print(f"configurations evaluated : {len(scores)}")
print(f"  best   (= best_score_) : {scores.max():.4f}   <- the number people report")
print(f"  median                 : {np.median(scores):.4f}")
print(f"  worst                  : {scores.min():.4f}")
print()
print(f"best is above the median by      : {scores.max() - np.median(scores):+.4f}")
print(f"fold-to-fold std of the winner   : {search.cv_results_['std_test_score'][search.best_index_]:.4f}")

configurations evaluated : 20
  best   (= best_score_) : 0.8508   <- the number people report
  median                 : 0.8436
  worst                  : 0.8121

best is above the median by      : +0.0072
fold-to-fold std of the winner   : 0.0077


**Read the last two lines together.** The winning configuration beats the median configuration by some margin — but the winner's own score varies across folds by a comparable amount.

That is the selection effect in miniature. Part of the winner's margin is real (some depths genuinely suit this data better) and part is luck (some configurations happened to fit these particular folds). **You cannot tell which part is which from inside the search**, because the search has no information the folds did not give it.

Reporting the maximum means reporting the real part *plus* the lucky part, with no way to separate them.


## 8.2 Worked example — intermediate: looking for the gap on our data, and not finding it

The obvious next move: compare `best_score_` against the untouched test set, and expect the first number to be higher.

Run it, and read the result carefully rather than the result you expect.


In [15]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score

optimistic = search.best_score_
test_auc = roc_auc_score(y_te, search.best_estimator_.predict_proba(X_te)[:, 1])
nested = cross_val_score(search, X_tr, y_tr,
                         cv=StratifiedKFold(3, shuffle=True, random_state=1),
                         scoring='roc_auc', n_jobs=-1)

print(f"(a) best_score_ from the search : {optimistic:.4f}")
print(f"(b) held-out test set           : {test_auc:.4f}")
print(f"(c) nested CV (3 outer folds)   : {nested.mean():.4f} +/- {nested.std():.4f}")
print()
print(f"gap (a - b) = {optimistic - test_auc:+.4f}")
print()
print("If that gap is NEGATIVE, the search's number came out BELOW the test number.")
print("That is not a broken demonstration. Read on.")

(a) best_score_ from the search : 0.8508
(b) held-out test set           : 0.8664
(c) nested CV (3 outer folds)   : 0.8401 +/- 0.0014

gap (a - b) = -0.0156

If that gap is NEGATIVE, the search's number came out BELOW the test number.
That is not a broken demonstration. Read on.


**The gap comes out negative.** On this data, `best_score_` lands *below* the test-set score. The effect this section is about does not show up here at all, and pretending otherwise would be dishonest.

Three reasons, all worth understanding:

**The test set is itself a noisy sample.** 2,400 rows with about 476 churners gives a test AUC with a standard error of roughly a hundredth. A gap of one or two hundredths in either direction is inside that noise. S3 taught confidence intervals precisely so that you would not read a difference this size as a finding.

**The selection effect here is small, because the search was small and the data is large.** Twenty configurations over 9,600 rows, with five folds each: every configuration's CV score is averaged over roughly 1,920 held-out rows. Well-averaged scores are not very noisy, so the maximum over 25 of them is not inflated by much — as §8.1 showed, the winner is barely above the median.

**Cross-validation trains on less data than the final refit.** Each CV fold trains on 80% of the training set. `best_estimator_` is then refitted on 100% of it before scoring against the test set. **More training data usually means a better model**, which pushes the test score up relative to the CV score — a systematic effect in the opposite direction to the selection effect.

That last point also explains line (c). Nested CV comes out **lowest** of the three, because its outer loop trains each inner search on only two-thirds of the data. Nested CV is unbiased *about the procedure*, but the procedure it measures is one trained on less data than you will actually use. It is the most rigorous number and mildly pessimistic — both at once.

**So: is the warning wrong?** No. It is conditional, and §8.3 supplies the condition.


## 8.3 Worked example — realistic: the regime where it bites

The selection effect scales with **how noisy each evaluation is** and **how many configurations you take the maximum over**. Our full dataset has quiet evaluations and a modest search, so nothing showed.

Change both. Cut the training data to 400 rows, and widen the search to 80 configurations. Nothing else changes.


In [16]:
small_idx = y_tr.sample(400, random_state=0).index
X_small, y_small = X_tr.loc[small_idx], y_tr.loc[small_idx]

wide = RandomizedSearchCV(
    Pipeline([('prep', prep), ('model', DecisionTreeClassifier(random_state=0))]),
    {'model__max_depth': np.arange(2, 31),
     'model__min_samples_leaf': np.arange(1, 121),
     'model__criterion': ['gini', 'entropy']},
    n_iter=80, cv=cv, scoring='roc_auc', random_state=0, n_jobs=-1).fit(X_small, y_small)

wide_test = roc_auc_score(y_te, wide.best_estimator_.predict_proba(X_te)[:, 1])

print("400 training rows, 80 configurations searched")
print(f"  best_score_ from the search : {wide.best_score_:.4f}   <- what you would report")
print(f"  held-out test set           : {wide_test:.4f}   <- what you would actually get")
print(f"  gap                         : {wide.best_score_ - wide_test:+.4f}")
print()
print(f"  compare: full data, 20 configs gave a gap of {optimistic - test_auc:+.4f}")

400 training rows, 80 configurations searched
  best_score_ from the search : 0.7854   <- what you would report
  held-out test set           : 0.7213   <- what you would actually get
  gap                         : +0.0641

  compare: full data, 20 configs gave a gap of -0.0156


**Now the gap is large and positive**, and it points the way the warning said it would.

Nothing about the model family changed. The data changed, and the search size changed. `best_score_` reported a number several points above what the model actually delivers on unseen customers, and an engineer who reported it would have overstated their model's performance by a margin that matters.

**The rule, now stated with its condition attached:**

> `best_score_` is optimistic by an amount that grows with the size of your search and shrinks with the size and cleanliness of your data. On 10,000 well-behaved rows with a 20-configuration search, the bias is small enough to disappear into noise. On a few hundred rows with a wide search, it is large enough to invalidate your report.
>
> Since you rarely know in advance which regime you are in, **report the number from data that took no part in selection.** It costs nothing and it is never wrong.

This is also a fair description of why public Kaggle leaderboards routinely overstate performance: thousands of competitors, each running enormous searches against the same public split, is precisely the high-search low-noise-budget regime — and the private leaderboard is the held-out test set doing its job.


### 🟢 Try It Yourself

Map the transition between the two regimes.

Run the §8.3 comparison at training sizes of 300, 600, 1,200, 2,400 and 9,600 rows, holding the search at 80 configurations, and plot the gap against training size.

Predict the shape before you run it.

<details>
<summary>Solution</summary>

```python
rows = []
for n in [300, 600, 1200, 2400, 9600]:
    idx = y_tr.sample(n, random_state=0).index
    s = RandomizedSearchCV(
        Pipeline([('prep', prep), ('model', DecisionTreeClassifier(random_state=0))]),
        {'model__max_depth': np.arange(2, 31),
         'model__min_samples_leaf': np.arange(1, 121),
         'model__criterion': ['gini', 'entropy']},
        n_iter=80, cv=cv, scoring='roc_auc', random_state=0, n_jobs=-1).fit(X_tr.loc[idx], y_tr.loc[idx])
    t = roc_auc_score(y_te, s.best_estimator_.predict_proba(X_te)[:, 1])
    rows.append({'n_train': n, 'best_score_': round(s.best_score_, 4),
                 'test AUC': round(t, 4), 'gap': round(s.best_score_ - t, 4)})
print(pd.DataFrame(rows).to_string(index=False))
```

Expect the gap to be clearly positive at the small sizes and to decay towards zero — and then into negative territory — as rows accumulate, where the refit-on-more-data effect from §8.2 takes over.

The transferable insight: **the optimism of `best_score_` is not a property of the model.** It is a property of your data size and your search size. Two engineers can report different "performance" for an identical model purely because one searched harder on less data.

</details>


# Common Pitfalls

| Pitfall | What happens | The fix |
|---|---|---|
| KNN or SVM without scaling | The largest-unit feature dominates the distance; the model silently ignores everything else | Always inside a `Pipeline` with `StandardScaler` |
| `GaussianNB` on one-hot columns | Fits a bell curve to a column of 0s and 1s | Use `CategoricalNB`, or accept it as a rough baseline only |
| `SVC` on a large dataset | Training time explodes; the cell appears to hang | `LinearSVC` or `SGDClassifier` above ~10k rows |
| Tuning `make_pipeline` steps | Grid keys break whenever you swap the model | Name your steps: `Pipeline([('prep', ...), ('model', ...)])` |
| Reporting `best_score_` | Optimistic by an unknown amount; inflates with search size | Report the held-out test score, or nested CV |
| Micro F1 on imbalanced multiclass | Identical to accuracy; hides total failure on rare classes | Report macro, plus the per-class table |
| Comparing searches with different `cv` or `scoring` | The comparison is meaningless | One `cv` object and one metric, defined before the searches |
| Tuning before checking features | Optimising the layer with the least headroom | Features first, model family second, hyperparameters last |

# FAQ

**Do I need to try every model family on every problem?**
No, and doing so is a mild red flag. For tabular data the defensible shortlist is a regularised linear model as a baseline and a tree-based model as the contender — which is Session 5. The map exists so you can *rule things out* quickly and explain why.

**`GridSearchCV` says `mean_test_score` — is that my test set?**
No. It is the mean across held-out CV folds inside the training set. The naming is unfortunate. Your test set is untouched until you deliberately touch it.

**How many configurations should I search?**
Start around 20–30 random draws. Look at whether the best score is meaningfully above the median draw. If every configuration scores about the same, the hyperparameters are not where your headroom is, and more searching will not find any.

**Should I tune on accuracy?**
Almost never on imbalanced data. Use `roc_auc` or `average_precision`. Session 6 makes the argument in full; for now, note that `scoring` is a decision, not a default to be accepted.

**My best model changes every time I re-run the search.**
That is informative. It means the configurations are within noise of each other. Check `std_test_score` — if the spread across folds exceeds the gap between configurations, they are tied, and you should pick the simplest one rather than the highest-scoring one.

**Why did `LogisticRegression` not need one-vs-rest?**
Modern sklearn fits multiclass logistic regression jointly with a softmax rather than wrapping binary models. You will meet softmax again in DL S1 as the standard classification output layer.

# Conclusion

You came into this notebook with two model families understood properly. You leave with a **map** — enough of KNN, Naive Bayes and SVM to shortlist sensibly and rule out confidently, without having spent six hours on algorithms you will rarely reach for.

The three things most worth keeping:

**Assumptions predict failures.** KNN assumes closeness means similarity, so it fails on unscaled and high-dimensional data. Naive Bayes assumes features are independent, so it distorts probabilities when they are not. SVM assumes a clean margin, so it struggles when classes overlap heavily and it scales badly with rows. You do not need the derivations to reason about any of that.

**Multiclass is mostly free, except for interpretation.** The split, the pipeline and the CV machinery did not change. The metrics and the coefficient story did.

**Tuning must be honest to be worth anything.** `best_score_` is a maximum over noisy attempts and is optimistic by construction. The number you report comes from data that took no part in selection. This is S3's discipline applied to a new decision, and it is the same reasoning as multiple comparisons in your statistics background.

# Transition to Notebook 04

Everything so far has been taught one piece at a time — a model here, a metric there, a search over here.

Notebook 04 is the **delivery block**: no new concepts, the full sequence assembled end to end on the churn data. Split, preprocess, cross-validate, tune, compare, and defend a decision with the S3 discipline attached.

That is the shape of the Employee Turnover CEP, rehearsed on data whose every trap was planted on purpose.

**Next:** open `ML_S4_04_practice_classification_pipeline.ipynb`.